In [1]:
import pandas as pd, numpy as np, json, pickle
from pathlib import Path
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor

ROOT = Path.cwd()
while not (ROOT/"outputs").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
OUT = ROOT/"outputs"

clean = pd.read_pickle(OUT/"clean.pkl")
CUTOFF = pd.Timestamp("2025-10-15")

def build_features(df):
    parts = []
    for loc, g in df.groupby(level="location"):
        g = g.copy()
        for L in (1,2,3,6,12,24): g[f"aqi_lag{L}"] = g.aqi_index.shift(L)
        for W in (6,24):
            g[f"aqi_roll{W}_mean"]=g.aqi_index.shift(1).rolling(W).mean()
            g[f"aqi_roll{W}_max"] =g.aqi_index.shift(1).rolling(W).max()
            g[f"aqi_roll{W}_std"] =g.aqi_index.shift(1).rolling(W).std()
        g["aqi_ewm12"]=g.aqi_index.shift(1).ewm(span=12).mean()
        g["aqi_mom3"]=g.aqi_lag1-g.aqi_lag3
        g["aqi_ratio24"]=g.aqi_lag1/(g["aqi_roll24_mean"]+1e-6)
        t=g.index.get_level_values("dt")
        g["hour"]=t.hour; g["month"]=t.month; g["dow"]=t.dayofweek
        g["is_weekend"]=(t.dayofweek>=5).astype(int)
        g["hour_sin"]=np.sin(2*np.pi*t.hour/24); g["hour_cos"]=np.cos(2*np.pi*t.hour/24)
        g["month_sin"]=np.sin(2*np.pi*t.month/12); g["month_cos"]=np.cos(2*np.pi*t.month/12)
        g["pressure_delta"]=g.pressure_mb.diff()
        g["wind_x_lag1"]=g.windspeed_kph*g.aqi_lag1
        g["target"]=g.aqi_index.shift(-1)
        g = g.drop(columns=["aqi_index","pm2_5","pm10","co","no2"])
        parts.append(g)
    return pd.concat(parts).dropna(subset=["target","aqi_lag24"])

feat = build_features(clean).reset_index()
tr = feat[feat.dt < CUTOFF]; te = feat[feat.dt >= CUTOFF]
drop = ["target","date_ist","time_ist","condition_text","description"]
Xtr = pd.get_dummies(tr.drop(columns=[c for c in drop if c in tr.columns]), columns=["location"]).select_dtypes(include=[np.number])
Xte = pd.get_dummies(te.drop(columns=[c for c in drop if c in te.columns]), columns=["location"]).select_dtypes(include=[np.number])
Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)

m = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6,
                 random_state=42, n_jobs=-1).fit(Xtr, tr.target)
pred_1h = m.predict(Xte)

pickle.dump(list(Xtr.columns), open(OUT/"path2_features.pkl", "wb"))
print(f"\nsaved → {OUT/'path2_features.pkl'}")
# Also 24h baseline
def build_24h(df):
    parts = []
    for loc, g in df.groupby(level="location"):
        g = g.copy()
        for L in (1,2,3,6,12,24): g[f"aqi_lag{L}"] = g.aqi_index.shift(L)
        for W in (6,24):
            g[f"aqi_roll{W}_mean"]=g.aqi_index.shift(1).rolling(W).mean()
        g["aqi_ewm12"]=g.aqi_index.shift(1).ewm(span=12).mean()
        g["aqi_mom3"]=g.aqi_lag1-g.aqi_lag3
        t=g.index.get_level_values("dt")
        g["hour"]=t.hour; g["month"]=t.month
        g["hour_sin"]=np.sin(2*np.pi*t.hour/24); g["hour_cos"]=np.cos(2*np.pi*t.hour/24)
        g["month_sin"]=np.sin(2*np.pi*t.month/12); g["month_cos"]=np.cos(2*np.pi*t.month/12)
        g["pressure_delta"]=g.pressure_mb.diff()
        g["windspeed_kph"]=g.windspeed_kph
        g["temp_c"]=g.temp_c; g["humidity"]=g.humidity
        g["target"]=g.aqi_index.shift(-24)
        g = g.drop(columns=["aqi_index","pm2_5","pm10","co","no2"])
        parts.append(g)
    return pd.concat(parts).dropna(subset=["target","aqi_lag24"])

feat24 = build_24h(clean).reset_index()
tr24 = feat24[feat24.dt < CUTOFF]; te24 = feat24[feat24.dt >= CUTOFF]
Xtr24 = pd.get_dummies(tr24.drop(columns=[c for c in drop if c in tr24.columns]), columns=["location"]).select_dtypes(include=[np.number])
Xte24 = pd.get_dummies(te24.drop(columns=[c for c in drop if c in te24.columns]), columns=["location"]).select_dtypes(include=[np.number])
Xte24 = Xte24.reindex(columns=Xtr24.columns, fill_value=0)
m24 = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6,
                   random_state=42, n_jobs=-1).fit(Xtr24, tr24.target)
pred_24h = m24.predict(Xte24)

baseline = {
    "snapshot_date": str(pd.Timestamp.now()),
    "train_rows_1h": len(Xtr), "test_rows_1h": len(Xte),
    "train_rows_24h": len(Xtr24), "test_rows_24h": len(Xte24),
    "baseline_1h_rmse": round(float(np.sqrt(((pred_1h-yte if False else pred_1h-te.target)**2).mean())), 2),
    "baseline_1h_r2": round(float(r2_score(te.target, pred_1h)), 4),
    "baseline_24h_rmse": round(float(np.sqrt(((pred_24h-te24.target)**2).mean())), 2),
    "baseline_24h_r2": round(float(r2_score(te24.target, pred_24h)), 4),
    "note": "Snapshot BEFORE Path 2. All Path 2 improvements compared to these numbers."
}

print(json.dumps(baseline, indent=2))
(OUT/"path2_baseline.json").write_text(json.dumps(baseline, indent=2))
print(f"\nsaved → {OUT/'path2_baseline.json'}")

C:\Users\hp\AppData\Local\Temp\ipykernel_7280\2451670408.py:27: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  g["hour"]=t.hour; g["month"]=t.month; g["dow"]=t.dayofweek
C:\Users\hp\AppData\Local\Temp\ipykernel_7280\2451670408.py:28: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  g["is_weekend"]=(t.dayofweek>=5).astype(int)
C:\Users\hp\AppData\Local\Temp\ipykernel_7280\2451670408.py:27: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  g["hour"]=t.hour; g["month"]=t.month; g["dow"]=t.dayofweek
C:\Users\hp\AppData\Local\Temp\ipykernel_7280\2451670408.py:28: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  g["is_weekend"]=(t.dayofweek>=5).asty


saved → c:\Users\hp\Downloads\research\outputs\path2_features.pkl
{
  "snapshot_date": "2026-11-13 00:24:23.282460",
  "train_rows_1h": 34281,
  "test_rows_1h": 11226,
  "train_rows_24h": 34281,
  "test_rows_24h": 11088,
  "baseline_1h_rmse": 3.58,
  "baseline_1h_r2": 0.9871,
  "baseline_24h_rmse": 35.93,
  "baseline_24h_r2": -0.2903,
  "note": "Snapshot BEFORE Path 2. All Path 2 improvements compared to these numbers."
}

saved → c:\Users\hp\Downloads\research\outputs\path2_baseline.json
